### Geocoding using Google Geocode API

In [8]:
##install shapely

!pip install shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 12.9 MB/s eta 0:00:00a 0:00:01


In [1]:
##setup
import pandas as pd
import numpy as np
import requests
import time

In [35]:
##import original files to geocode
original_geocode = pd.read_csv('./data/geocoding_original.csv')
manual_geocode = pd.read_csv('./data/Civil Summons Geocoding.csv')

In [30]:
##import new files to geocode (updated when september data was added)
new_geocode = pd.read_csv('./data/Full_Cleaned_Unmatch_Geocode.csv')
new_manual_geocode = pd.read_csv('./data/geocoding2.csv')

In [37]:
##check column names
original_geocode.columns

Index(['Ticket_Number', 'O_Violation_Location_Borough',
       'O_Violation_Location_Block_No.', 'O_Violation_Location_Lot_No',
       'O_Violation_Location_House_No.', 'O_Violation_Location_Street_Name',
       'O_Violation_Location_City', 'O_Violation_Location_Zip_Code',
       'O_Violation_Location_State_Name', 'Category (Gemini)',
       'Revised_Address (Gemini)'],
      dtype='object')

In [39]:
##check column names
manual_geocode.columns

Index(['Ticket_Number', 'O_Violation_Location_Borough',
       'O_Violation_Location_Block_No.', 'O_Violation_Location_Lot_No',
       'O_Violation_Location_House_No.', 'O_Violation_Location_Street_Name',
       'O_Violation_Location_City', 'O_Violation_Location_Zip_Code',
       'O_Violation_Location_State_Name', 'Category (Gemini)',
       'Revised_Address (Gemini)', 'Updated_Address', 'Ticket used',
       'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16',
       'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20',
       'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24',
       'Unnamed: 25'],
      dtype='object')

In [32]:
##check column names
new_geocode.columns

Index(['Ticket_Number', 'O_violation_location_borough',
       'O_violation_location_block_no', 'O_violation_location_lot_no',
       'O_violation_location_house_no', 'O_violation_location_street_name',
       'O_violation_location_city', 'O_violation_location_zip_code',
       'O_violation_location_state', 'Category (Gemini)',
       'Revised_Address (Gemini)'],
      dtype='object')

In [34]:
##check column names
new_manual_geocode.columns

Index(['Ticket_Number', 'O_violation_location_borough',
       'O_violation_location_block_no', 'O_violation_location_lot_no',
       'O_violation_location_house_no', 'O_violation_location_street_name',
       'O_violation_location_city', 'O_violation_location_zip_code',
       'O_violation_location_state', 'Category (Gemini)',
       'Revised_Address (Gemini)', 'Updated_Address', 'Ticket used'],
      dtype='object')

In [41]:
##drop unnecessary columns
original_geocode.drop(['Category (Gemini)', 'O_Violation_Location_Block_No.', 'O_Violation_Location_Lot_No',
                       'O_Violation_Location_House_No.', 'O_Violation_Location_Street_Name'], axis = 1, inplace = True)

In [43]:
##check column names again
original_geocode.columns

Index(['Ticket_Number', 'O_Violation_Location_Borough',
       'O_Violation_Location_City', 'O_Violation_Location_Zip_Code',
       'O_Violation_Location_State_Name', 'Revised_Address (Gemini)'],
      dtype='object')

In [47]:
##drop unnecessary columns
manual_geocode.drop(['Category (Gemini)', 'O_Violation_Location_Block_No.', 'O_Violation_Location_Lot_No',
                     'O_Violation_Location_House_No.', 'O_Violation_Location_Street_Name', 'Ticket used', 
                     'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16',
                     'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22',
                     'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25'], axis = 1, inplace = True)

In [49]:
##check column names again
manual_geocode.columns

Index(['Ticket_Number', 'O_Violation_Location_Borough',
       'O_Violation_Location_City', 'O_Violation_Location_Zip_Code',
       'O_Violation_Location_State_Name', 'Revised_Address (Gemini)',
       'Updated_Address'],
      dtype='object')

In [36]:
##drop unnecessary columns
new_geocode.drop(['Category (Gemini)', 'O_violation_location_block_no', 'O_violation_location_lot_no',
                       'O_violation_location_house_no', 'O_violation_location_street_name'], axis = 1, inplace = True)

In [38]:
##check column names again
new_geocode.columns

Index(['Ticket_Number', 'O_violation_location_borough',
       'O_violation_location_city', 'O_violation_location_zip_code',
       'O_violation_location_state', 'Revised_Address (Gemini)'],
      dtype='object')

In [40]:
##drop unnecessary columns
new_manual_geocode.drop(['Category (Gemini)', 'O_violation_location_block_no', 'O_violation_location_lot_no',
                       'O_violation_location_house_no', 'O_violation_location_street_name', 'Ticket used'], axis = 1, inplace = True)

In [42]:
##check column names again
new_manual_geocode.columns

Index(['Ticket_Number', 'O_violation_location_borough',
       'O_violation_location_city', 'O_violation_location_zip_code',
       'O_violation_location_state', 'Revised_Address (Gemini)',
       'Updated_Address'],
      dtype='object')

In [51]:
##merge the gemini geocoded file with the manual geocoded file by ticket number
geocode_main = original_geocode.merge(
    manual_geocode[['Ticket_Number', 'Updated_Address']],
    on='Ticket_Number',
    how='left'
)

# Overwrite target_col only for rows that had a match (non-NaN new_col)
mask = geocode_main['Updated_Address'].notna()
geocode_main.loc[mask, 'Revised_Address (Gemini)'] = geocode_main.loc[mask, 'Updated_Address']

# Drop the helper column
geocode_main.drop('Updated_Address', axis = 1, inplace = True)

In [44]:
##merge the gemini geocoded file with the manual geocoded file by ticket number
geocode_main_new = new_geocode.merge(
    new_manual_geocode[['Ticket_Number', 'Updated_Address']],
    on='Ticket_Number',
    how='left'
)

# Overwrite target_col only for rows that had a match (non-NaN new_col)
mask = geocode_main_new['Updated_Address'].notna()
geocode_main_new.loc[mask, 'Revised_Address (Gemini)'] = geocode_main_new.loc[mask, 'Updated_Address']

# Drop the helper column
geocode_main_new.drop('Updated_Address', axis = 1, inplace = True)

In [53]:
##check column names
geocode_main.columns

Index(['Ticket_Number', 'O_Violation_Location_Borough',
       'O_Violation_Location_City', 'O_Violation_Location_Zip_Code',
       'O_Violation_Location_State_Name', 'Revised_Address (Gemini)'],
      dtype='object')

In [55]:
##check geocoded file
geocode_main.head()

,Ticket_Number,O_Violation_Location_Borough,O_Violation_Location_City,O_Violation_Location_Zip_Code,O_Violation_Location_State_Name,Revised_Address (Gemini)
0,190851430,BRONX,BRONX,10457,NEW YORK,2176 TIEBOUT AVENUE
1,196184708,BROOKLYN,BROOKLYN,11226,NEW YORK,1050 OCEAN AVENUE
2,196241834,QUEENS,QUEENS,.,NEW YORK,CENTRAL AVE AND BEACH
3,196242292,QUEENS,QUEENS,.,NEW YORK,BEACH CHANNEL DRIVE AND MOTT AVE
4,196244840,QUEENS,FAR ROCKAWAY,11691,NEW YORK,19-15 SEAGIRT BOULEVARD


In [46]:
##check column names
geocode_main_new.columns

Index(['Ticket_Number', 'O_violation_location_borough',
       'O_violation_location_city', 'O_violation_location_zip_code',
       'O_violation_location_state', 'Revised_Address (Gemini)'],
      dtype='object')

In [57]:
##replace periods with NA
geocode_main = geocode_main.replace('.', pd.NA)

In [48]:
##replace periods with NA
geocode_main_new = geocode_main_new.replace('.', pd.NA)

In [59]:
##view geocoded file
geocode_main.head()

,Ticket_Number,O_Violation_Location_Borough,O_Violation_Location_City,O_Violation_Location_Zip_Code,O_Violation_Location_State_Name,Revised_Address (Gemini)
0,190851430,BRONX,BRONX,10457,NEW YORK,2176 TIEBOUT AVENUE
1,196184708,BROOKLYN,BROOKLYN,11226,NEW YORK,1050 OCEAN AVENUE
2,196241834,QUEENS,QUEENS,<NA>,NEW YORK,CENTRAL AVE AND BEACH
3,196242292,QUEENS,QUEENS,<NA>,NEW YORK,BEACH CHANNEL DRIVE AND MOTT AVE
4,196244840,QUEENS,FAR ROCKAWAY,11691,NEW YORK,19-15 SEAGIRT BOULEVARD


In [65]:
##access the google geocoding API
API_KEY = ##YOUR API KEY
url = "https://maps.googleapis.com/maps/api/geocode/json"

# Store results
geocode_main['oath_latitude'] = None
geocode_main['oath_longitude'] = None

##loop through the addresses in the file, combining each part of the address column
for idx, row in geocode_main.iterrows():
    address_parts = [
    row['Revised_Address (Gemini)'],
    row['O_Violation_Location_City'],
    row['O_Violation_Location_State_Name'],
    str(row['O_Violation_Location_Zip_Code']) if pd.notna(row['O_Violation_Location_Zip_Code']) else None
    ]
    address = ', '.join(part for part in address_parts if pd.notna(part) and part)
    
    params = {
        "address": address,
        "key": API_KEY
    }

    ##access the API
    try:
        response = requests.get(url, params=params).json()
        
        if response["results"]:
            location = response["results"][0]["geometry"]["location"]
            geocode_main.at[idx, 'oath_latitude'] = location["lat"]
            geocode_main.at[idx, 'oath_longitude'] = location["lng"]
        else:
            print(f"No results for row {idx}: {address}")
            
    except Exception as e:
        print(f"Error on row {idx}: {e}")
    
    time.sleep(0.05)  # stay within API rate limits

No results for row 1997: WEST 47 ST AND STREET BROADWAY
No results for row 2184: 
No results for row 2341: 7TH AVE AND 31 ST
No results for row 2594: WEST 40 ST AND STREET 7 AVENUE
No results for row 2692: 31 ST AND 7 AVENUE
No results for row 4418: 
No results for row 4431: W 15 ST AND ST BWAY


In [50]:
##access the google geocoding API
API_KEY = ##YOUR API KEY
url = "https://maps.googleapis.com/maps/api/geocode/json"

# Store results
geocode_main_new['oath_latitude'] = None
geocode_main_new['oath_longitude'] = None

##loop through the addresses in the file, combining each part of the address column
for idx, row in geocode_main_new.iterrows():
    address_parts = [
    row['Revised_Address (Gemini)'],
    row['O_violation_location_city'],
    row['O_violation_location_state'],
    str(row['O_violation_location_zip_code']) if pd.notna(row['O_violation_location_zip_code']) else None
    ]
    address = ', '.join(part for part in address_parts if pd.notna(part) and part)
    
    params = {
        "address": address,
        "key": API_KEY
    }
    ##access the API
    try:
        response = requests.get(url, params=params).json()
        
        if response["results"]:
            location = response["results"][0]["geometry"]["location"]
            geocode_main_new.at[idx, 'oath_latitude'] = location["lat"]
            geocode_main_new.at[idx, 'oath_longitude'] = location["lng"]
        else:
            print(f"No results for row {idx}: {address}")
            
    except Exception as e:
        print(f"Error on row {idx}: {e}")
    
    time.sleep(0.05)

No results for row 218: 38 ST AND ST
No results for row 219: S E 33 ST AND ST
No results for row 222: 32 ST AND ST
No results for row 249: 
No results for row 250: 
No results for row 367: WEST AND ST
No results for row 371: 13 ST AND ST
No results for row 372: 13 ST AND ST
No results for row 393: WEST AND ST
No results for row 394: WEST AND ST
No results for row 402: WEST AND ST
No results for row 404: 43 ST AND ST
No results for row 425: 8 AVE AND AVE
No results for row 461: WEST AND ST


In [67]:
##view file with lat/long
geocode_main.head()

,Ticket_Number,O_Violation_Location_Borough,O_Violation_Location_City,O_Violation_Location_Zip_Code,O_Violation_Location_State_Name,Revised_Address (Gemini),oath_latitude,oath_longitude
0,190851430,BRONX,BRONX,10457,NEW YORK,2176 TIEBOUT AVENUE,40.854515,-73.898188
1,196184708,BROOKLYN,BROOKLYN,11226,NEW YORK,1050 OCEAN AVENUE,40.636648,-73.958646
2,196241834,QUEENS,QUEENS,<NA>,NEW YORK,CENTRAL AVE AND BEACH,40.60719,-73.74954
3,196242292,QUEENS,QUEENS,<NA>,NEW YORK,BEACH CHANNEL DRIVE AND MOTT AVE,40.605348,-73.755315
4,196244840,QUEENS,FAR ROCKAWAY,11691,NEW YORK,19-15 SEAGIRT BOULEVARD,40.594817,-73.753899


In [69]:
##export to csv
geocode_main.to_csv('oath_geocoded.csv', index = False)

In [52]:
##view file with new lat/long fields
geocode_main_new.head()

,Ticket_Number,O_violation_location_borough,O_violation_location_city,O_violation_location_zip_code,O_violation_location_state,Revised_Address (Gemini),oath_latitude,oath_longitude
0,196178960,BROOKLYN,BROOKLYN,11215.0,NEW YORK,PROSPECT PARK WEST AND FLATBUSH AVE,40.675021,-73.971115
1,196184735,BROOKLYN,BROOKLYN,NaN,NEW YORK,980 CONEY,40.575544,-73.970702
2,196373504,MANHATTAN,NEW YORK,NaN,NEW YORK,7TH AVE AND AVE,40.766948,-73.979051
3,196374916,MANHATTAN,NEW YORK,10030.0,NEW YORK,2310 7 AVENUE,40.815982,-73.94371
4,196374925,MANHATTAN,NEW YORK,10030.0,NEW YORK,2310 7 AVENUE,40.815982,-73.94371


In [57]:
##rename columns in the new geocoded file
geocode_main_new.rename(columns={"O_violation_location_borough": "O_Violation_Location_Borough", 
                                 "O_violation_location_city": "O_Violation_Location_City",
                                "O_violation_location_zip_code": "O_Violation_Location_Zip_Code",
                                "O_violation_location_state": "O_Violation_Location_State_Name"}, inplace=True)

In [59]:
##export to csv
geocode_main_new.to_csv('oath_geocoded_new.csv', index = False)

#### Geocode Updates

In [4]:
##import previously geocoded data and civil summons v5
geocoded_old = pd.read_csv('./data/oath_geocoded.csv')
updated_citations = pd.read_csv('./data/civil_summons_2021_v5.csv')

In [6]:
##merge and keep only non-matches
merged = pd.merge(geocoded_old, updated_citations, on="Ticket_Number", how='outer', indicator=True)

unmatched = merged[merged['_merge'] != 'both']

In [8]:
##view the unmatched rows
unmatched.head()

,Ticket_Number,O_Violation_Location_Borough_x,O_Violation_Location_City_x,O_Violation_Location_Zip_Code_x,O_Violation_Location_State_Name_x,Revised_Address (Gemini),oath_latitude,oath_longitude,Final_Officer_Tax_ID,Officer.first.name,...,CCRB_fado_Abuse of Authority,CCRB_fado_Discourtesy,CCRB_fado_Force,CCRB_fado_Offensive Language,CCRB_fado_Untruthful Statement,Reg.hours,Overtime.hours,Last.active,yrs_of_service,_merge
5,196178960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,960116,KAREEM,...,0.0,0.0,0.0,0.0,0.0,2080.0,235.42,20301.0,10,right_only
9,196184735,NaN,NaN,NaN,NaN,NaN,NaN,NaN,961743,PATRICIA,...,1.0,1.0,0.0,0.0,0.0,2080.0,256.83,20301.0,9,right_only
34,196373504,NaN,NaN,NaN,NaN,NaN,NaN,NaN,969559,ANDRES,...,0.0,0.0,2.0,0.0,0.0,1280.0,31.83,20301.0,5,right_only
36,196374916,NaN,NaN,NaN,NaN,NaN,NaN,NaN,953738,DENIS,...,6.0,1.0,2.0,0.0,0.0,2080.0,542.85,20301.0,13,right_only
37,196374925,NaN,NaN,NaN,NaN,NaN,NaN,NaN,960171,ANTHONY,...,2.0,0.0,0.0,0.0,0.0,2080.0,594.05,20301.0,10,right_only


In [10]:
##see total unmatched rows
len(unmatched)

492

In [12]:
##drop unnecessary columns
unmatched.drop(["O_Violation_Location_Borough_x", "O_Violation_Location_City_x", "O_Violation_Location_Zip_Code_x", 
                "O_Violation_Location_State_Name_x", "Revised_Address (Gemini)", "oath_latitude", "oath_longitude"],
               axis = 1, inplace = True)
unmatched.head()

/var/folders/gv/sd1_fqg951l514f5hk_w0ls80000gn/T/ipykernel_89412/3102086367.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unmatched.drop(["O_Violation_Location_Borough_x", "O_Violation_Location_City_x", "O_Violation_Location_Zip_Code_x",


,Ticket_Number,Final_Officer_Tax_ID,Officer.first.name,Officer.last.name,cs_first,cs_middle,cs_last,pr_first,pr_middle,pr_last,...,CCRB_fado_Abuse of Authority,CCRB_fado_Discourtesy,CCRB_fado_Force,CCRB_fado_Offensive Language,CCRB_fado_Untruthful Statement,Reg.hours,Overtime.hours,Last.active,yrs_of_service,_merge
5,196178960,960116,KAREEM,ABDELAZIZ,KAREEM,K,ABDELAZIZ,KAREEM,K,ABDELAZIZ,...,0.0,0.0,0.0,0.0,0.0,2080.0,235.42,20301.0,10,right_only
9,196184735,961743,PATRICIA,ENNIS,PATRICIA,NaN,ENNIS,PATRICIA,NaN,ENNIS,...,1.0,1.0,0.0,0.0,0.0,2080.0,256.83,20301.0,9,right_only
34,196373504,969559,ANDRES,CADAVID,ANDRES,NaN,CADAVID,ANDRES,NaN,CADAVID,...,0.0,0.0,2.0,0.0,0.0,1280.0,31.83,20301.0,5,right_only
36,196374916,953738,DENIS,CEKIC,DENIS,NaN,CEKIC,DENIS,NaN,CEKIC,...,6.0,1.0,2.0,0.0,0.0,2080.0,542.85,20301.0,13,right_only
37,196374925,960171,ANTHONY,ARENA,ANTHONY,J,ARENA,ANTHONY,J,ARENA,...,2.0,0.0,0.0,0.0,0.0,2080.0,594.05,20301.0,10,right_only


In [14]:
##export to csv
unmatched.to_csv('./data/unmatched.csv', index = False)

#### Merge old and new geocoded datasets

In [67]:
##import old file
geocode_main = pd.read_csv('./data/oath_geocoded.csv')

In [69]:
##get the number of rows of the old file
len(geocode_main)

4665

In [71]:
##get the number of rows of the new file
len(geocode_main_new)

492

In [73]:
##concatenate files
combined_geocode = pd.concat([geocode_main, geocode_main_new], ignore_index=True)

In [75]:
##get the new length
len(combined_geocode)

5157

In [77]:
##export to csv
combined_geocode.to_csv('./data/combined_geocode.csv', index = False)

#### Add Census Tracts

In [79]:
##setup
import geopandas as gpd
from shapely import wkt

In [246]:
##import file and view
geocoded = pd.read_csv('./data/combined_geocode.csv')
geocoded.head()

,Ticket_Number,O_Violation_Location_Borough,O_Violation_Location_City,O_Violation_Location_Zip_Code,O_Violation_Location_State_Name,Revised_Address (Gemini),oath_latitude,oath_longitude
0,190851430,BRONX,BRONX,10457.0,NEW YORK,2176 TIEBOUT AVENUE,40.854515,-73.898188
1,196184708,BROOKLYN,BROOKLYN,11226.0,NEW YORK,1050 OCEAN AVENUE,40.636648,-73.958646
2,196241834,QUEENS,QUEENS,NaN,NEW YORK,CENTRAL AVE AND BEACH,40.607190,-73.749540
3,196242292,QUEENS,QUEENS,NaN,NEW YORK,BEACH CHANNEL DRIVE AND MOTT AVE,40.605348,-73.755315
4,196244840,QUEENS,FAR ROCKAWAY,11691.0,NEW YORK,19-15 SEAGIRT BOULEVARD,40.594817,-73.753899


In [248]:
##import the census tracts file and view
tracts = pd.read_csv('./data/2020_Census_Blocks_nyc.csv')
tracts.head()

,the_geom,CB2020,BoroCode,BoroName,CT2020,BCTCB2020,GEOID,Shape_Length,Shape_Area
0,MULTIPOLYGON (((-74.03995040788483 40.70089063...,1000,1,Manhattan,100,10001001000,360610001001000,"6,437.85374521","1,202,838.17013"
1,MULTIPOLYGON (((-74.04387761639944 40.69018767...,1001,1,Manhattan,100,10001001001,360610001001001,"4,395.19018343","640,166.352288"
2,MULTIPOLYGON (((-73.98495042073655 40.71235553...,2000,1,Manhattan,201,10002012000,360610002012000,"2,055.29576202","263,308.402003"
3,MULTIPOLYGON (((-73.9799619880113 40.713972699...,1000,1,Manhattan,202,10002021000,360610002021000,"1,187.88411083","57,115.9366413"
4,MULTIPOLYGON (((-73.98187417339794 40.71144411...,2000,1,Manhattan,202,10002022000,360610002022000,"2,178.77742688","157,313.542923"


In [250]:
##convert the_geom to geometry not string

tracts["geometry"] = tracts["the_geom"].apply(wkt.loads)

# Create GeoDataFrame
tracts_geom = gpd.GeoDataFrame(tracts, geometry="geometry", crs="EPSG:4326")

In [252]:
##pull out points

points = gpd.GeoDataFrame(
    geocoded,
    geometry=gpd.points_from_xy(geocoded['oath_longitude'], geocoded['oath_latitude']),
    crs="EPSG:4326"
)

In [254]:
##spatial join

full_with_census = gpd.sjoin(points, tracts_geom, how="left", predicate="within")

In [256]:
##view the full file
full_with_census.head()

,Ticket_Number,O_Violation_Location_Borough,O_Violation_Location_City,O_Violation_Location_Zip_Code,O_Violation_Location_State_Name,Revised_Address (Gemini),oath_latitude,oath_longitude,geometry,index_right,the_geom,CB2020,BoroCode,BoroName,CT2020,BCTCB2020,GEOID,Shape_Length,Shape_Area
0,190851430,BRONX,BRONX,10457.0,NEW YORK,2176 TIEBOUT AVENUE,40.854515,-73.898188,POINT (-73.898 40.855),6421.0,MULTIPOLYGON (((-73.89600716111786 40.85559634...,1002.0,2.0,Bronx,38303.0,2.038303e+10,3.600504e+14,"2,473.14950724","221,630.244551"
1,196184708,BROOKLYN,BROOKLYN,11226.0,NEW YORK,1050 OCEAN AVENUE,40.636648,-73.958646,POINT (-73.959 40.637),27019.0,MULTIPOLYGON (((-73.95839961041402 40.63632693...,1002.0,3.0,Brooklyn,51800.0,3.051800e+10,3.604705e+14,"1,797.75427279","178,580.200882"
2,196241834,QUEENS,QUEENS,NaN,NEW YORK,CENTRAL AVE AND BEACH,40.607190,-73.749540,POINT (-73.75 40.607),11993.0,MULTIPOLYGON (((-73.74869062731368 40.60787392...,1009.0,4.0,Queens,103202.0,4.103202e+10,3.608110e+14,"1,756.57017236","138,527.354464"
3,196242292,QUEENS,QUEENS,NaN,NEW YORK,BEACH CHANNEL DRIVE AND MOTT AVE,40.605348,-73.755315,POINT (-73.755 40.605),15759.0,MULTIPOLYGON (((-73.75516471628974 40.60830903...,2005.0,4.0,Queens,103201.0,4.103201e+10,3.608110e+14,"3,493.74180212","437,336.945331"
4,196244840,QUEENS,FAR ROCKAWAY,11691.0,NEW YORK,19-15 SEAGIRT BOULEVARD,40.594817,-73.753899,POINT (-73.754 40.595),17234.0,MULTIPOLYGON (((-73.75398602683511 40.59514008...,3000.0,4.0,Queens,99802.0,4.099802e+10,3.608110e+14,"2,483.91672811","312,395.432574"


In [258]:
##get the number of NAs
full_with_census['index_right'].isna().sum()

50

In [260]:
# Filter rows where the point did NOT match any tract
no_tract = full_with_census[full_with_census['index_right'].isna()]

##export to csv
no_tract.to_csv('no_census_match.csv', index = False)

In [262]:
##update columns
geocode_census = full_with_census[['Ticket_Number', 'oath_latitude', 'oath_longitude', 'CB2020', 'CT2020', 'BCTCB2020']]
geocode_census.head()

,Ticket_Number,oath_latitude,oath_longitude,CB2020,CT2020,BCTCB2020
0,190851430,40.854515,-73.898188,1002.0,38303.0,2.038303e+10
1,196184708,40.636648,-73.958646,1002.0,51800.0,3.051800e+10
2,196241834,40.607190,-73.749540,1009.0,103202.0,4.103202e+10
3,196242292,40.605348,-73.755315,2005.0,103201.0,4.103201e+10
4,196244840,40.594817,-73.753899,3000.0,99802.0,4.099802e+10


In [264]:
##export to csv
geocode_census.to_csv('./data/geocode_census.csv', index = False)

### Merge with Full Dataset

In [266]:
##import file
oath_full = pd.read_csv('./data/civil_summons_2021_v5.csv')

In [268]:
##import file
geocoded_tickets = pd.read_csv('./data/geocode_census.csv')

In [270]:
##view the file
oath_full.head()

,Ticket_Number,Final_Officer_Tax_ID,Officer.first.name,Officer.last.name,cs_first,cs_middle,cs_last,pr_first,pr_middle,pr_last,...,CCRB_Unknown_Allegations,CCRB_fado_Abuse of Authority,CCRB_fado_Discourtesy,CCRB_fado_Force,CCRB_fado_Offensive Language,CCRB_fado_Untruthful Statement,Reg.hours,Overtime.hours,Last.active,yrs_of_service
0,190851430,953430,NICHOLAS,SORECA,NICHOLAS,V,SORECA,NICHOLAS,V,SORECA,...,0.0,3.0,0.0,0.0,0.0,0.0,2080.0,100.95,20301.0,13
1,196184708,961743,PATRICIA,ENNIS,PATRICIA,NaN,ENNIS,PATRICIA,NaN,ENNIS,...,0.0,1.0,1.0,0.0,0.0,0.0,2080.0,256.83,20301.0,9
2,196241834,961465,JAMES,WYNNE,JAMES,W,WYNNE,JAMES,W,WYNNE,...,0.0,1.0,0.0,0.0,0.0,0.0,2080.0,182.67,20301.0,10
3,196242292,964085,MICHAEL,KENJESKY,MICHAEL,V,KENJESKY,MICHAEL,V,KENJESKY,...,0.0,0.0,0.0,1.0,0.0,0.0,2080.0,46.42,20301.0,8
4,196244840,964778,MICHAEL,SINTO,MICHAEL,G,SINTO,MICHAEL,G,SINTO,...,0.0,4.0,0.0,0.0,0.0,0.0,2080.0,379.42,20301.0,8


In [272]:
##get the length
len(oath_full)

5155

In [274]:
##view
geocoded_tickets.head()

,Ticket_Number,oath_latitude,oath_longitude,CB2020,CT2020,BCTCB2020
0,190851430,40.854515,-73.898188,1002.0,38303.0,2.038303e+10
1,196184708,40.636648,-73.958646,1002.0,51800.0,3.051800e+10
2,196241834,40.607190,-73.749540,1009.0,103202.0,4.103202e+10
3,196242292,40.605348,-73.755315,2005.0,103201.0,4.103201e+10
4,196244840,40.594817,-73.753899,3000.0,99802.0,4.099802e+10


In [276]:
##get length
len(geocoded_tickets)

5157

In [278]:
##get the number of duplicate tickets
print(geocoded_tickets["Ticket_Number"].duplicated().sum())

23


In [280]:
##get duplicates
dupes = geocoded_tickets[geocoded_tickets["Ticket_Number"].duplicated(keep=False)]
print(dupes.sort_values("Ticket_Number"))

      Ticket_Number  oath_latitude  oath_longitude  CB2020    CT2020  \
1214      196357958      40.663960      -73.939929  2000.0   33100.0   
908       196357958      40.663960      -73.939929  2000.0   33100.0   
1301      198742620      40.627855      -74.137720  2008.0   24700.0   
909       198742620      40.627855      -74.137720  2008.0   24700.0   
1310      198928795      40.674995      -73.856532  4001.0    5800.0   
1309      198928795      40.674995      -73.856532  4001.0    5800.0   
910       198928795      40.674995      -73.856532  4001.0    5800.0   
911       198928795      40.674995      -73.856532  4001.0    5800.0   
1314      198982044      40.733908      -73.872589  2005.0   68300.0   
912       198982044      40.733908      -73.872589  2005.0   68300.0   
1330      199225694      40.763560      -73.836007  2008.0   86900.0   
901       199225694      40.763560      -73.836007  2008.0   86900.0   
1331      199228352      40.753330      -73.829764  1000.0   797

In [282]:
##remove duplicated tickets
geocoded_tickets_deduped = geocoded_tickets.drop_duplicates(subset="Ticket_Number")

In [284]:
##merge the oath file with the deduplicated geocoded file
oath_geocode = pd.merge(oath_full, geocoded_tickets_deduped, on="Ticket_Number", how='left')
oath_geocode.head()

,Ticket_Number,Final_Officer_Tax_ID,Officer.first.name,Officer.last.name,cs_first,cs_middle,cs_last,pr_first,pr_middle,pr_last,...,CCRB_fado_Untruthful Statement,Reg.hours,Overtime.hours,Last.active,yrs_of_service,oath_latitude,oath_longitude,CB2020,CT2020,BCTCB2020
0,190851430,953430,NICHOLAS,SORECA,NICHOLAS,V,SORECA,NICHOLAS,V,SORECA,...,0.0,2080.0,100.95,20301.0,13,40.854515,-73.898188,1002.0,38303.0,2.038303e+10
1,196184708,961743,PATRICIA,ENNIS,PATRICIA,NaN,ENNIS,PATRICIA,NaN,ENNIS,...,0.0,2080.0,256.83,20301.0,9,40.636648,-73.958646,1002.0,51800.0,3.051800e+10
2,196241834,961465,JAMES,WYNNE,JAMES,W,WYNNE,JAMES,W,WYNNE,...,0.0,2080.0,182.67,20301.0,10,40.607190,-73.749540,1009.0,103202.0,4.103202e+10
3,196242292,964085,MICHAEL,KENJESKY,MICHAEL,V,KENJESKY,MICHAEL,V,KENJESKY,...,0.0,2080.0,46.42,20301.0,8,40.605348,-73.755315,2005.0,103201.0,4.103201e+10
4,196244840,964778,MICHAEL,SINTO,MICHAEL,G,SINTO,MICHAEL,G,SINTO,...,0.0,2080.0,379.42,20301.0,8,40.594817,-73.753899,3000.0,99802.0,4.099802e+10


In [286]:
##get length
len(oath_geocode)

5155

In [288]:
##export to csv
oath_geocode.to_csv('./data/oath_with_geocode_v1.csv', index = False)

#### Add respondent race for 2021

In [290]:
##setup
import glob

In [292]:
##get csv files for 2021 data
path = './data/2021_files' # change to your path
all_files = glob.glob(path + "/*.csv")

# 2. Read each file and combine them into one DataFrame
combined_2021 = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)

combined_2021.head()

,Ticket Number,Complaintant's First Name,Complaintant's Last Name,Tax Registry Number/Tax ID,Agency,Confidence on PO Name,Confidence on Officer Tax ID,Alternative Officer First Name,Alternative Officer Last Name,Alternative Officer Tax ID,...,Computer Generated (Y/N),If Computer Generated,Respondent Race,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,198169869.0,.,Lewis,967576,NYPD,High,High,.,.,.,...,N,.,White,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,198171775.0,.,Michel,949956,NYPD,High,High,.,.,.,...,N,.,Hisp. Black,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,198203648.0,.,Cadoo,963424,NYPD,High,Medium,.,.,963429,...,N,.,Hisp. Black,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,198204198.0,Illegible,Illegible,954143,NYPD,.,High,.,.,.,...,N,.,Black,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,198221495.0,.,Lowe,966176,NYPD,High,High,.,.,.,...,N,.,Black,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [294]:
##get the datatype
oath_geocode['Ticket_Number'].dtype

dtype('int64')

In [296]:
##get column name
combined_2021.columns

Index(['Ticket Number ', 'Complaintant's First Name',
       'Complaintant's Last Name', 'Tax Registry Number/Tax ID', 'Agency',
       'Confidence on PO Name', 'Confidence on Officer Tax ID',
       'Alternative Officer First Name', 'Alternative Officer Last Name',
       'Alternative Officer Tax ID', 'Verified Name w/Tax ID',
       'Last Name, First Name', 'If Partial/Illegible ID',
       'Final Officer First Name', 'Final Officer Last Name',
       'Final Officer Tax ID', 'Computer Generated (Y/N)',
       'If Computer Generated', 'Respondent Race', 'Unnamed: 19',
       'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23',
       'Unnamed: 24', 'Unnamed: 25'],
      dtype='object')

In [298]:
##get datatype
combined_2021['Ticket Number '].dtype

dtype('float64')

In [300]:
##change to int64 datatype
combined_2021['Ticket Number '] = combined_2021['Ticket Number '].astype('Int64')

In [302]:
##view file
combined_2021.head()

,Ticket Number,Complaintant's First Name,Complaintant's Last Name,Tax Registry Number/Tax ID,Agency,Confidence on PO Name,Confidence on Officer Tax ID,Alternative Officer First Name,Alternative Officer Last Name,Alternative Officer Tax ID,...,Computer Generated (Y/N),If Computer Generated,Respondent Race,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,198169869,.,Lewis,967576,NYPD,High,High,.,.,.,...,N,.,White,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,198171775,.,Michel,949956,NYPD,High,High,.,.,.,...,N,.,Hisp. Black,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,198203648,.,Cadoo,963424,NYPD,High,Medium,.,.,963429,...,N,.,Hisp. Black,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,198204198,Illegible,Illegible,954143,NYPD,.,High,.,.,.,...,N,.,Black,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,198221495,.,Lowe,966176,NYPD,High,High,.,.,.,...,N,.,Black,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [304]:
##filter to only have race and ticket number
combined_2021 = combined_2021[['Ticket Number ', 'Respondent Race']]
combined_2021.head()

,Ticket Number,Respondent Race
0,198169869,White
1,198171775,Hisp. Black
2,198203648,Hisp. Black
3,198204198,Black
4,198221495,Black


In [306]:
##get column names
oath_geocode.columns

Index(['Ticket_Number', 'Final_Officer_Tax_ID', 'Officer.first.name',
       'Officer.last.name', 'cs_first', 'cs_middle', 'cs_last', 'pr_first',
       'pr_middle', 'pr_last', 'Officer_Total_Summonses', 'month_issued',
       'Respondent_Race', 'roster_rank', 'roster_command', 'roster_race',
       'roster_gender', 'O_Violation_Date', 'O_Violation_Time',
       'O_Issuing_Agency', 'O_Respondent_first_name', 'O_Respondent_last_name',
       'O_Balance_Due', 'O_Violation_Location_Borough',
       'O_Violation_Location_City', 'O_Violation_Location_Zip_Code',
       'O_Violation_Location_State_Name', 'O_Respondent_Address_Borough',
       'O_Hearing_Status', 'O_Hearing_Result', 'O_Scheduled_Hearing_Location',
       'O_Hearing_Date', 'O_Hearing_Time', 'O_Decision_Location_Borough',
       'O_Decision_Date', 'O_Total_Violation_Amount', 'O_Violation_Details',
       'O_Date_Judgment_Docketed', 'O_Penalty_Imposed', 'O_Paid_Amount',
       'O_Add_Penalties_Late_Fees', 'O_Compliance_Status',
 

In [308]:
##drop respondent race column in original file
oath_geocode.drop(columns=['Respondent_Race'], inplace=True)

In [310]:
##merge original file and the race file on ticket number
race_merged = pd.merge(oath_geocode, combined_2021, left_on='Ticket_Number', right_on='Ticket Number ', how='left')

##view file
race_merged.head()

,Ticket_Number,Final_Officer_Tax_ID,Officer.first.name,Officer.last.name,cs_first,cs_middle,cs_last,pr_first,pr_middle,pr_last,...,Overtime.hours,Last.active,yrs_of_service,oath_latitude,oath_longitude,CB2020,CT2020,BCTCB2020,Ticket Number,Respondent Race
0,190851430,953430,NICHOLAS,SORECA,NICHOLAS,V,SORECA,NICHOLAS,V,SORECA,...,100.95,20301.0,13,40.854515,-73.898188,1002.0,38303.0,2.038303e+10,190851430,.
1,196184708,961743,PATRICIA,ENNIS,PATRICIA,NaN,ENNIS,PATRICIA,NaN,ENNIS,...,256.83,20301.0,9,40.636648,-73.958646,1002.0,51800.0,3.051800e+10,196184708,Asian/Pacific Is.
2,196241834,961465,JAMES,WYNNE,JAMES,W,WYNNE,JAMES,W,WYNNE,...,182.67,20301.0,10,40.607190,-73.749540,1009.0,103202.0,4.103202e+10,196241834,.
3,196242292,964085,MICHAEL,KENJESKY,MICHAEL,V,KENJESKY,MICHAEL,V,KENJESKY,...,46.42,20301.0,8,40.605348,-73.755315,2005.0,103201.0,4.103201e+10,196242292,Black
4,196244840,964778,MICHAEL,SINTO,MICHAEL,G,SINTO,MICHAEL,G,SINTO,...,379.42,20301.0,8,40.594817,-73.753899,3000.0,99802.0,4.099802e+10,196244840,Black


In [312]:
##get the length
len(race_merged)

5198

In [314]:
##check for duplicated
race_merged['Ticket_Number'].duplicated().sum()

64

In [316]:
##drop duplicates
race_merged = race_merged.drop_duplicates(subset=['Ticket_Number'])

In [318]:
##get length
len(race_merged)

5134

In [320]:
##drop column
race_merged.drop(['Ticket Number '], axis = 1, inplace= True)

In [322]:
##export to csv
race_merged.to_csv('./data/civil_summons_2021_v6.csv', index = False)